# EyePACS adaptation and the EyePACS-adapted APTOS arms (record §69, §69.2)

**Nothing in this notebook runs unless its flag in cell [2] is set to `True` — and the long steps also need their
confirmation phrase.** Running all cells with the defaults verifies the inputs and prints the state of every run.

| step | flag | what it does | cost |
|---|---|---|---|
| A0 | `STAGE_CACHE` / `VERIFY_CACHE` | copy the EyePACS frame cache Drive → local disk; verify it against the split | ~55 GB copy |
| A1 | `RUN_SPEED_CHECK` | ten optimiser steps of a throwaway model at batch 16; saves nothing | minutes |
| A  | `RUN_ADAPTATION` | **the one** EyePACS adaptation run of P (seed 42); pins the selected checkpoint | many hours |
| B  | `RUN_HELDOUT_TEST` | scores the pinned model once on the 16,249 held-out EyePACS test images | ~1 h |
| C  | `RUN_APTOS_P_EP` | P-EP: APTOS seeds 42 → 123 → 2026 from the pinned encoder, fresh CORN head each | hours |
| D  | `RUN_APTOS_E1_EP` | E1-EP: the same with E1's lesion head (λ = 1), after P-EP | hours |
| E  | `RUN_E2_EP` | E2-EP — **only** if the E1-EP probe criterion against P-EP is recorded as met | hours |

**Protocol (fixed in §69 / §69.2; not editable here).** Adaptation: P's model, weighted CORN with the EyePACS training
class weights, AdamW 1e-4, weight decay 0.05, batch 16, at most 50 epochs, early stopping 12, selection by EyePACS
validation QWK only; no APTOS file is read. The held-out EyePACS test images select nothing. APTOS arms: the exact P / E1
protocol (batch 2, the pre-registered APTOS class weights, the 2,921 / 730 split, BEST by APTOS validation QWK); the only
difference from P / E1 is the encoder they start from. The frame caches are built on the laptop
(`python eyepacs_adaptation_data.py build --cache-dir <dir>`, and `--set heldout-test` for the test images) and uploaded.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import e1_data as ed, e1_train as e1t, e2_control as e2
import eyepacs_adaptation_data as ea, eyepacs_adaptation_train as eat, ep_aptos_train as ep
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SWITCHES -- everything expensive is OFF. Set a flag to True (and type its phrase) to allow that step. ====
STAGE_CACHE      = False   # A0: copy the adaptation frame cache from Drive to /content (about 55 GB)
VERIFY_CACHE     = False   # A0: verify a cache already on /content (re-hashes every shard)
RUN_SPEED_CHECK  = False   # A1: ten steps of a throwaway model; nothing is saved, nothing is chosen from it
RUN_ADAPTATION   = False   # A : THE EyePACS adaptation run (one run, seed 42)        -- many hours on a T4
RUN_HELDOUT_TEST = False   # B : the pinned model on the held-out EyePACS test images -- once
RUN_APTOS_P_EP   = False   # C : P-EP on APTOS, three seeds
RUN_APTOS_E1_EP  = False   # D : E1-EP on APTOS, three seeds (after P-EP)
RUN_E2_EP        = False   # E : E2-EP -- only with a recorded, met E1-EP probe criterion
CONFIRM = {"adaptation": "", "heldout_test": "", "p_ep": "", "e1_ep": "", "e2_ep": ""}
PHRASES = {"adaptation": "RUN THE ONE EYEPACS ADAPTATION", "heldout_test": "SCORE THE EYEPACS TEST SET ONCE",
           "p_ep": "TRAIN P-EP ON APTOS", "e1_ep": "TRAIN E1-EP ON APTOS", "e2_ep": "TRAIN E2-EP ON APTOS"}

def confirmed(step, flag):
    if flag and CONFIRM[step] != PHRASES[step]:
        raise RuntimeError(f"{step}: the flag is True but CONFIRM['{step}'] is not the phrase {PHRASES[step]!r}")
    return bool(flag)
print({k: v for k, v in globals().items() if k.startswith(("RUN_", "STAGE_", "VERIFY_"))})

In [ ]:
# ==== [3] INPUTS AND PROTOCOL (reads manifests and small files only; trains nothing) ====
EXP = colab_config.DRIVE.experiments_root
SPLIT_DIR = posixpath.join(REPO_DIR, "dataset_splits")
ROWS = ea.read_split(SPLIT_DIR, ea.MANIFEST_SHA256)                      # refuses any other manifest
SPLIT = ea.split_report(ROWS)
assert SPLIT["images"] == {"train": 31610, "val": 3512, "total": 35122} and SPLIT["patients_in_both_splits"] == 0
assert not {r["image"] for r in ROWS} & set(ea.BLANK_TRAINING_IMAGES)
COUNTS, EYEPACS_CLASS_WEIGHTS = eat.class_weights(ROWS)                  # must equal the recorded weights
TEST_ROWS = ea.read_test_manifest(SPLIT_DIR, ea.TEST_MANIFEST_SHA256)
assert len(TEST_ROWS) == 16249 and ea.assert_disjoint_patients(TEST_ROWS, ROWS)
assert eat.PROTOCOL == dict(at.P_PROTOCOL, batch_size=16) and (eat.ADAPTATION_RUNS, eat.ADAPTATION_SEED) == (1, 42)
assert ep.PROTOCOL is at.P_PROTOCOL and ep.SEEDS == (42, 123, 2026) and e1t.LAMBDA == 1.0
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256
DRIVE_CACHE = posixpath.join(colab_config.DATASET_ROOT, "EyePACS", "adaptation_frames_v1")
DRIVE_TEST_CACHE = posixpath.join(colab_config.DATASET_ROOT, "EyePACS", "heldout_test_frames_v1")
LOCAL_CACHE, LOCAL_TEST_CACHE = "/content/eyepacs_adaptation_frames_v1", "/content/eyepacs_heldout_test_frames_v1"
ADAPTATION_DIR = eat.run_dir_for(EXP, ea.MANIFEST_SHA256)
ELIGIBILITY = posixpath.join(EXP, ep.EXPERIMENT, ep.ELIGIBILITY_NAME)
print("adaptation split", ea.MANIFEST_SHA256[:16], SPLIT["images"], "| training counts", COUNTS)
print("EyePACS class weights", [round(w, 4) for w in EYEPACS_CLASS_WEIGHTS])
print("adaptation protocol", eat.PROTOCOL, "| runs:", eat.ADAPTATION_RUNS, "| seed:", eat.ADAPTATION_SEED)
print("held-out test", ea.TEST_MANIFEST_SHA256[:16], len(TEST_ROWS), "images -- evaluation only")
print("APTOS arms", {a: ("mandatory" if v["mandatory"] else "conditional") for a, v in ep.ARMS.items()}, "| seeds", ep.SEEDS)
print("adaptation run:", eat.state(ADAPTATION_DIR), "|", ADAPTATION_DIR)
print("pinned:", os.path.exists(posixpath.join(ADAPTATION_DIR, eat.FROZEN_NAME)), "| held-out evaluated:", eat.heldout_result(ADAPTATION_DIR) is not None,
      "| E2-EP criterion recorded:", os.path.exists(ELIGIBILITY))

In [ ]:
# ==== [4] A0 -- FRAME CACHE: Drive -> local disk (every shard checked by SHA-256), verified against the split ====
FRAMES = None
if STAGE_CACHE:
    print(ea.stage_cache(DRIVE_CACHE, LOCAL_CACHE))
if STAGE_CACHE or VERIFY_CACHE:
    # staging hashes every shard as it copies; VERIFY_CACHE re-hashes a cache that is already local
    FRAMES = ea.FrameCache(LOCAL_CACHE, ROWS, check_hashes=bool(VERIFY_CACHE), log=print)
    print(json.dumps(FRAMES.report, indent=1))
    assert FRAMES.report["frames"] == {"train": 31610, "val": 3512}
    assert FRAMES.report["identity"]["manifest_sha256"] == ea.MANIFEST_SHA256 and FRAMES.report["identity"]["dtype"] == ea.FRAME_DTYPE
else:
    print("frame cache not staged (STAGE_CACHE = VERIFY_CACHE = False)")

In [ ]:
# ==== [5] A1 -- SPEED / MEMORY CHECK: ten optimiser steps of a THROWAWAY model at batch 16. ====
# Not part of the experiment: nothing is saved, no checkpoint or run directory is touched, nothing is chosen from it.
if RUN_SPEED_CHECK:
    import time
    assert FRAMES is not None, "stage or verify the cache first (cell 4)"
    _STEPS, _BATCH = 10, eat.PROTOCOL["batch_size"]
    tf.keras.backend.clear_session()
    _, _arrays = pl.load_reference(PRETRAINED)
    _model = eat.build_compiled_model(eat.ADAPTATION_SEED, _arrays, EYEPACS_CLASS_WEIGHTS)
    _seq = eat.make_epoch_sequence(FRAMES, FRAMES.entries("train")[:_STEPS * _BATCH], 0, eat.ADAPTATION_SEED, _BATCH, augment=True)
    _load, _step = [], []
    for _b in range(_STEPS):
        _t = time.time(); _x, _y = _seq[_b]; _load.append(time.time() - _t)
        _t = time.time(); _model.train_on_batch(_x, _y); _step.append(time.time() - _t)
    _s, _l = float(np.median(_step[2:])), float(np.median(_load))
    _n = int(np.ceil(31610 / _BATCH))
    _mem = tf.config.experimental.get_memory_info("GPU:0")
    print(f"batch {_BATCH}: {_s:.2f} s/step ({_BATCH / _s:.1f} images/s) + {_l:.2f} s data loading per batch (single thread)")
    print(f"steps per epoch {_n} | training part of an epoch ~{_n * _s / 60:.0f} min (plus validation of 3,512 images)")
    print(f"peak GPU memory {_mem['peak'] / 2**30:.1f} GiB")
    del _model, _arrays, _seq
    tf.keras.backend.clear_session()
else:
    print("speed check not run (RUN_SPEED_CHECK = False)")

In [ ]:
# ==== [6] A -- THE EYEPACS ADAPTATION RUN (one run, seed 42). THIS TRAINS for many hours. Resumable. ====
if confirmed("adaptation", RUN_ADAPTATION):
    assert FRAMES is not None, "stage or verify the cache first (cell 4)"
    tf.keras.backend.clear_session()
    _, IMAGENET_ARRAYS = pl.load_reference(PRETRAINED)
    ADAPTATION = eat.run_adaptation(EXP, FRAMES, IMAGENET_ARRAYS, EYEPACS_CLASS_WEIGHTS, repo_dir=REPO_DIR,
                                    staging_root="/content/eyepacs_adaptation_staging")
    _f = ADAPTATION["frozen"]
    print(f"EyePACS validation QWK {_f['val_qwk']:.4f} at epoch index {_f['best_epoch_index']} of {_f['epochs_run']} "
          f"({ADAPTATION['stop']['stop_reason']}) | pinned {_f['sha256']}")
else:
    print("adaptation not run (RUN_ADAPTATION = False):", eat.state(ADAPTATION_DIR))

In [ ]:
# ==== [7] B -- HELD-OUT EYEPACS TEST: the PINNED model, once. Selects nothing; read by no other step. ====
if confirmed("heldout_test", RUN_HELDOUT_TEST):
    print(ea.stage_cache(DRIVE_TEST_CACHE, LOCAL_TEST_CACHE))
    TEST_FRAMES = ea.FrameCache(LOCAL_TEST_CACHE, TEST_ROWS, ea.TEST_MANIFEST_SHA256, check_hashes=False, log=print)
    assert TEST_FRAMES.report["frames"] == {"test": 16249}
    tf.keras.backend.clear_session()
    _, IMAGENET_ARRAYS = pl.load_reference(PRETRAINED)
    HELDOUT = eat.evaluate_heldout_test(ADAPTATION_DIR, TEST_FRAMES, IMAGENET_ARRAYS, repo_dir=REPO_DIR)
    print(HELDOUT["role"])
    print({k: v for k, v in HELDOUT["metrics"].items() if not isinstance(v, (dict, list))})
else:
    print("held-out test not run (RUN_HELDOUT_TEST = False); recorded:", eat.heldout_result(ADAPTATION_DIR) is not None)

In [ ]:
# ==== [8] APTOS INPUTS for the EP arms (only when an APTOS step is switched on): bundle, weights, pinned encoder ====
_APTOS = any((confirmed("p_ep", RUN_APTOS_P_EP), confirmed("e1_ep", RUN_APTOS_E1_EP), confirmed("e2_ep", RUN_E2_EP)))
if _APTOS:
    EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # the Stage-4 teacher of E1 (§47/§48)
    GEN4 = cache.stage4_generation_id(EXPECTED_MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
    BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
    DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
    LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2_e1", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
    ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=EXPECTED_MODEL_SHA)
    ed.stage_inputs(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
    BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                            expected_stage4_sha256=EXPECTED_MODEL_SHA, roots=LOCAL_ROOTS)
    ed.require_inputs(BUNDLE, BUNDLE.population)
    _tr, _va, _split_sha = msr.verify_split()                             # the authoritative APTOS split
    _empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
    assert _split_sha == v2cfg.SPLIT_SHA256 == BUNDLE.bundle["split_sha256"]
    assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
    assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
    SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
    PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
    APTOS_CLASS_WEIGHTS = list(PREREG["class_weights"])
    at.assert_protocol(PREREG, APTOS_CLASS_WEIGHTS)                       # P's protocol and P's class weights
    # E1's recorded lesion prior (the training-target means its lesion-head bias was initialised from), unchanged
    _E1_CONFIGS = [json.load(open(posixpath.join(e1t.run_dir_for(EXP, EXPECTED_MODEL_SHA, s), "config.json"))) for s in ep.SEEDS]
    LESION_PRIOR = _E1_CONFIGS[0]["lesion_prior"]
    assert all(c["lesion_prior"] == LESION_PRIOR and c["lesion_loss_weight"] == 1.0 for c in _E1_CONFIGS)
    tf.keras.backend.clear_session()
    _, IMAGENET_ARRAYS = pl.load_reference(PRETRAINED)
    # refuses unless the adaptation checkpoint is pinned AND its held-out evaluation is recorded (order of §69.2 D)
    ADAPTED = ep.load_adapted_encoder(ADAPTATION_DIR, IMAGENET_ARRAYS)
    print("pinned adapted encoder", ADAPTED["sha256"], "| differs from ImageNet:", ADAPTED["digest"] != ADAPTED["imagenet_digest"])
    print("lesion prior (E1's)", [round(p, 4) for p in LESION_PRIOR], "| APTOS class weights", [round(w, 4) for w in APTOS_CLASS_WEIGHTS])
    for _arm in ep.ARMS:
        print(" ", _arm, {s: ep.seed_state(ep.run_dir_for(EXP, _arm, ADAPTED["sha256"], s)) for s in ep.SEEDS})
else:
    print("no APTOS step is switched on")

In [ ]:
# ==== [9] C -- P-EP on APTOS: seeds 42 -> 123 -> 2026, each from the SAME pinned encoder with a fresh CORN head ====
if confirmed("p_ep", RUN_APTOS_P_EP):
    P_EP = ep.run_arm("p_ep", EXP, BUNDLE, ADAPTED, APTOS_CLASS_WEIGHTS, repo_dir=REPO_DIR, staging_root="/content/ep_aptos_staging")
    print({s: round(r["best"]["metrics"]["qwk"], 4) for s, r in P_EP.items()})
else:
    print("P-EP not run (RUN_APTOS_P_EP = False)")

In [ ]:
# ==== [10] D -- E1-EP on APTOS (mandatory, after P-EP): E1 unchanged, started from the same pinned encoder ====
if confirmed("e1_ep", RUN_APTOS_E1_EP):
    E1_EP = ep.run_arm("e1_ep", EXP, BUNDLE, ADAPTED, APTOS_CLASS_WEIGHTS, lesion_prior=LESION_PRIOR, repo_dir=REPO_DIR,
                       staging_root="/content/ep_aptos_staging")
    print({s: round(r["best"]["metrics"]["qwk"], 4) for s, r in E1_EP.items()})
else:
    print("E1-EP not run (RUN_APTOS_E1_EP = False)")

In [ ]:
# ==== [11] E -- E2-EP: CONDITIONAL. Refused unless the E1-EP vs P-EP probe criterion is recorded as met. ====
if confirmed("e2_ep", RUN_E2_EP):
    print("criterion record:", ep.require_e2_ep_eligibility(ELIGIBILITY))  # raises if absent or not met
    E2_PARTNER = e2.derangement(BUNDLE.train_ids)                         # the fixed derangement of E2 (seed 20261001)
    E2_EP = ep.run_arm("e2_ep", EXP, BUNDLE, ADAPTED, APTOS_CLASS_WEIGHTS, lesion_prior=LESION_PRIOR, partner=E2_PARTNER,
                       eligibility=ELIGIBILITY, repo_dir=REPO_DIR, staging_root="/content/ep_aptos_staging")
    print({s: round(r["best"]["metrics"]["qwk"], 4) for s, r in E2_EP.items()})
else:
    print("E2-EP not run (RUN_E2_EP = False); criterion recorded:", os.path.exists(ELIGIBILITY))